In [19]:
import numpy as np


In [ ]:
class MultiHeadAttention:

    def __init__(self, token_dim, head_num, use_wo=True):
        self.head_num = head_num
        self.token_dim = token_dim
        self.d = token_dim
        self.use_wo = use_wo

        if token_dim % head_num != 0:
            raise ValueError('head_num must be divisible by token_dim')

        self.head_dim = token_dim // head_num
        bound = np.sqrt(3 / token_dim)
        self.Wq = np.random.uniform(-bound, bound, (token_dim, token_dim))
        self.Wk = np.random.uniform(-bound, bound, (token_dim, token_dim))
        self.Wv = np.random.uniform(-bound, bound, (token_dim, token_dim))

        self.Bq = np.zeros(token_dim)
        self.Bk = np.zeros(token_dim)
        self.Bv = np.zeros(token_dim)

        # momentumn for optimizer
        self.Wq_momentumn = np.zeros_like(self.Wq)
        self.Wk_momentumn = np.zeros_like(self.Wk)
        self.Wv_momentumn = np.zeros_like(self.Wv)
        self.Bq_momentumn = np.zeros_like(self.Bq)
        self.Bk_momentumn = np.zeros_like(self.Bk)
        self.Bv_momentumn = np.zeros_like(self.Bv)

        # output projection Wo, Bo
        if self.use_wo:
            self.Wo = np.random.uniform(-bound, bound, (token_dim, token_dim))
            self.Bo = np.zeros(token_dim)
            self.Wo_momentumn = np.zeros_like(self.Wo)
            self.Bo_momentumn = np.zeros_like(self.Bo)

    def softmax(self, z):
        exp = np.exp(z - np.max(z, axis=-1, keepdims=True))
        return exp / np.sum(exp, axis=-1, keepdims=True)

    def forward(self, input, mask=None):
        self.input = input
        self.inputs = input
        self.mask = mask
        batch_size, seq_len, _ = input.shape
        self.batch_size = batch_size
        self.seq_len = seq_len

        # linear projections
        Q = input @ self.Wq + self.Bq
        K = input @ self.Wk + self.Bk
        V = input @ self.Wv + self.Bv

        # reshape and transpose: batch_size, head_num, seq_len, head_dim
        Q = Q.reshape(batch_size, seq_len, self.head_num, self.head_dim).transpose(0, 2, 1, 3)
        K = K.reshape(batch_size, seq_len, self.head_num, self.head_dim).transpose(0, 2, 1, 3)
        V = V.reshape(batch_size, seq_len, self.head_num, self.head_dim).transpose(0, 2, 1, 3)

        self.Q = Q
        self.K = K
        self.V = V
        # batch_size, head_num, seq_len, head_dim

        # scaled dot-product attention: batch_size * head_num * seq_len * seq_len
        scores = (Q @ K.transpose(0, 1, 3, 2)) / np.sqrt(self.head_dim)
        if mask is not None:
            scores = np.where(mask == 0, -1e9, scores)
        self.scores = scores

        Z = self.softmax(scores)
        self.Z = Z
        # batch_size * head_num * seq_len * seq_len

        O = Z @ V
        self.O = O
        # batch_size * head_num * seq_len * head_dim

        # concat heads: batch_size * seq_len * token_dim
        O = O.transpose(0, 2, 1, 3)
        concat = O.reshape(batch_size, seq_len, self.d)
        self.concat = concat

        # output projection
        if self.use_wo:
            self.output = concat @ self.Wo + self.Bo
        else:
            self.output = concat

        return self.output

    def backward(self, dvalues):
        batch_size = self.batch_size
        seq_len = self.seq_len

        # 1. gradient through output projection Wo, Bo
        if self.use_wo:
            concat_flat = self.concat.reshape(-1, self.d)
            dvalues_flat = dvalues.reshape(-1, self.d)
            self.d_Wo = concat_flat.T @ dvalues_flat
            self.d_Bo = np.sum(dvalues, axis=(0, 1))
            d_concat = dvalues @ self.Wo.T
        else:
            d_concat = dvalues

        # 2. reshape and transpose d_concat: batch_size, head_num, seq_len, head_dim
        d_O = d_concat.reshape(batch_size, seq_len, self.head_num, self.head_dim).transpose(0, 2, 1, 3)

        # 3. gradient w.r.t V and Z (from O = Z @ V)
        # batch_size, head_num, seq_len, head_dim
        d_V = self.Z.transpose(0, 1, 3, 2) @ d_O
        # batch_size, head_num, seq_len, seq_len
        d_Z = d_O @ self.V.transpose(0, 1, 3, 2)

        # 4. gradient through softmax
        sum_dZ_Z = np.sum(d_Z * self.Z, axis=-1, keepdims=True)
        d_scores = self.Z * (d_Z - sum_dZ_Z)

        # gradient through mask
        if self.mask is not None:
            d_scores = np.where(self.mask == 0, 0, d_scores)

        # 5. gradient through scale factor sqrt(head_dim)
        d_scores = d_scores / np.sqrt(self.head_dim)

        # 6. gradient w.r.t Q and K (from scores = Q @ K.T)
        d_Q = d_scores @ self.K
        d_K = d_scores.transpose(0, 1, 3, 2) @ self.Q

        # 7. reshape and transpose d_Q, d_K, d_V back to (batch_size, seq_len, self.d)
        d_Q_flat = d_Q.transpose(0, 2, 1, 3).reshape(batch_size, seq_len, self.d)
        d_K_flat = d_K.transpose(0, 2, 1, 3).reshape(batch_size, seq_len, self.d)
        d_V_flat = d_V.transpose(0, 2, 1, 3).reshape(batch_size, seq_len, self.d)

        # 8. gradient w.r.t weights Wq, Wk, Wv and biases Bq, Bk, Bv
        input_flat = self.input.reshape(-1, self.d)
        self.d_Wq = input_flat.T @ d_Q_flat.reshape(-1, self.d)
        self.d_Wk = input_flat.T @ d_K_flat.reshape(-1, self.d)
        self.d_Wv = input_flat.T @ d_V_flat.reshape(-1, self.d)

        self.d_Bq = np.sum(d_Q_flat, axis=(0, 1))
        self.d_Bk = np.sum(d_K_flat, axis=(0, 1))
        self.d_Bv = np.sum(d_V_flat, axis=(0, 1))

        # 9. gradient w.r.t inputs (sum of grad Q, K, V )
        d_Iq = d_Q_flat @ self.Wq.T
        d_Ik = d_K_flat @ self.Wk.T
        d_Iv = d_V_flat @ self.Wv.T

        self.d_inputs = d_Iq + d_Ik + d_Iv
        self.d_input = self.d_inputs

        return self.d_inputs


In [21]:
class mha_optimizer:
    def __init__(self, lr=0.001, beta=0.9):
        self.lr = lr
        self.beta = beta

    def OptimizerGD(self, layer):
        layer.Wq -= self.lr * layer.d_Wq
        layer.Wk -= self.lr * layer.d_Wk
        layer.Wv -= self.lr * layer.d_Wv
        layer.Bq -= self.lr * layer.d_Bq
        layer.Bk -= self.lr * layer.d_Bk
        layer.Bv -= self.lr * layer.d_Bv
        if layer.use_wo:
            layer.Wo -= self.lr * layer.d_Wo
            layer.Bo -= self.lr * layer.d_Bo

    def OptimizerGD_WithMomentumn(self, layer):
        layer.Wq_momentumn = self.beta * layer.Wq_momentumn - self.lr * layer.d_Wq
        layer.Wk_momentumn = self.beta * layer.Wk_momentumn - self.lr * layer.d_Wk
        layer.Wv_momentumn = self.beta * layer.Wv_momentumn - self.lr * layer.d_Wv
        layer.Bq_momentumn = self.beta * layer.Bq_momentumn - self.lr * layer.d_Bq
        layer.Bk_momentumn = self.beta * layer.Bk_momentumn - self.lr * layer.d_Bk
        layer.Bv_momentumn = self.beta * layer.Bv_momentumn - self.lr * layer.d_Bv

        layer.Wq += layer.Wq_momentumn
        layer.Wk += layer.Wk_momentumn
        layer.Wv += layer.Wv_momentumn
        layer.Bq += layer.Bq_momentumn
        layer.Bk += layer.Bk_momentumn
        layer.Bv += layer.Bv_momentumn

        if layer.use_wo:
            layer.Wo_momentumn = self.beta * layer.Wo_momentumn - self.lr * layer.d_Wo
            layer.Bo_momentumn = self.beta * layer.Bo_momentumn - self.lr * layer.d_Bo
            layer.Wo += layer.Wo_momentumn
            layer.Bo += layer.Bo_momentumn


In [ ]:
# CONFIG
batch_size = 2
seq_len = 5
token_dim = 16
head_num = 4

# input embeddings
np.random.seed(42)
inputs = np.random.randn(batch_size, seq_len, token_dim)
print("input shape:", inputs.shape)

# instantiate mha
mha = MultiHeadAttention(token_dim=token_dim, head_num=head_num, use_wo=True)

# forward pass
output = mha.forward(inputs)
print("output shape:", output.shape)

#  gradient from next layer (dvalues)
dvalues = np.random.randn(*output.shape)

# backward pass
d_inputs = mha.backward(dvalues)
print("d_inputs shape:", d_inputs.shape)
print("d_Wq shape:", mha.d_Wq.shape)
print("d_Wk shape:", mha.d_Wk.shape)
print("d_Wv shape:", mha.d_Wv.shape)
if mha.use_wo:
    print("d_Wo shape:", mha.d_Wo.shape)


Input shape: (2, 5, 16)
Output shape: (2, 5, 16)
d_inputs shape: (2, 5, 16)
d_Wq shape: (16, 16)
d_Wk shape: (16, 16)
d_Wv shape: (16, 16)
d_Wo shape: (16, 16)


In [23]:
# check grad
eps = 1e-6
numerical_dinputs = np.zeros_like(inputs)

for idx in np.ndindex(inputs.shape):
    inputs_plus = inputs.copy()
    inputs_plus[idx] += eps
    loss_plus = np.sum(mha.forward(inputs_plus) * dvalues)

    inputs_minus = inputs.copy()
    inputs_minus[idx] -= eps
    loss_minus = np.sum(mha.forward(inputs_minus) * dvalues)

    numerical_dinputs[idx] = (loss_plus - loss_minus) / (2 * eps)

mha.forward(inputs)
mha.backward(dvalues)

rel_error = np.max(np.abs(mha.d_inputs - numerical_dinputs) / (np.maximum(1e-7, np.abs(mha.d_inputs) + np.abs(numerical_dinputs))))
print(f"gradient check d_inputs relative error: {rel_error:.2e}")
assert rel_error < 1e-5, "gradient check failed"
print("gradient check passed")


gradient check d_inputs relative error: 1.05e-07
gradient check passed


In [24]:
# Test
opt = mha_optimizer(lr=0.01, beta=0.9)
opt.OptimizerGD_WithMomentumn(mha)
print("weight updated")


weight updated
